In [1]:
import os
import glob
import cv2
from ultralytics import YOLO
import pandas as pd
import numpy as np
from collections import defaultdict

In [2]:
def compute_iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    
    inter_area = max(0, x2 - x1 + 1) * max(0, y2 - y1 + 1)
    box1_area = (box1[2] - box1[0] + 1) * (box1[3] - box1[1] + 1)
    box2_area = (box2[2] - box2[0] + 1) * (box2[3] - box2[1] + 1)

    union_area = box1_area + box2_area - inter_area
    return inter_area / union_area if union_area > 0 else 0

In [3]:
def load_predictions(pred_dir):
    pred_dict = defaultdict(list)
    for file in glob.glob(os.path.join(pred_dir, "*.txt")):
        filename = os.path.splitext(os.path.basename(file))[0]
        try:
            frame = int(filename.split("_")[-1])
        except ValueError:
            print(f"Skipping file {filename}, could not parse frame ID.")
            continue

        with open(file, "r") as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 6:
                    continue
                cls, x_min, y_min, x_max, y_max, conf = parts
                box = [float(x_min), float(y_min), float(x_max), float(y_max)]
                pred_dict[frame].append((box, float(conf)))
    return pred_dict

In [4]:
def load_ground_truth(csv_path):
    df = pd.read_csv(csv_path)
    gt_dict = defaultdict(list)
    for _, row in df.iterrows():
        frame = int(row["Frame"])
        box = [row["min_x"], row["min_y"], row["max_x"], row["max_y"]]
        gt_dict[frame].append(box)
    return gt_dict

In [5]:
def evaluate_at_iou(gt_dict, pred_dict, iou_thresh=0.5):
    TP, FP, FN = 0, 0, 0

    for frame in gt_dict.keys():
        gt_boxes = gt_dict[frame]
        preds = sorted(pred_dict.get(frame, []), key=lambda x: -x[1])

        matched = set()
        for box_pred, conf in preds:
            ious = [compute_iou(box_pred, box_gt) for box_gt in gt_boxes]
            max_iou = max(ious) if ious else 0
            best_gt = np.argmax(ious) if ious else -1

            if max_iou >= iou_thresh and best_gt not in matched:
                TP += 1
                matched.add(best_gt)
            else:
                FP += 1

        FN += len(gt_boxes) - len(matched)

    precision = TP / (TP + FP + 1e-6)
    recall = TP / (TP + FN + 1e-6)
    f1 = 2 * precision * recall / (precision + recall + 1e-6)

    return precision, recall, f1

In [6]:
def evaluate_map(gt_dict, pred_dict):
    iou_thresholds = np.arange(0.5, 1.0, 0.05)
    aps = []

    for iou in iou_thresholds:
        precision, recall, _ = evaluate_at_iou(gt_dict, pred_dict, iou)
        aps.append(precision)

    return np.mean(aps), aps

In [7]:
def run_yolov8(model, video_path):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return None

    try:
        class_names = model.names
        name_to_id = {v: k for k, v in class_names.items()}
        car_class_id = name_to_id['car']
    except (AttributeError, KeyError):
        car_class_id = 2

    pred_dict = defaultdict(list)
    frame_number = 0

    while cap.isOpened():
        success, frame = cap.read()
        if not success:
            break

        results = model(frame)
        result = results[0]

        boxes = result.boxes.xyxy.cpu().numpy()
        confs = result.boxes.conf.cpu().numpy()
        class_ids = result.boxes.cls.cpu().numpy()

        for box, conf, cls_id in zip(boxes, confs, class_ids):
            if int(cls_id) == car_class_id:
                pred_dict[frame_number].append((box.tolist(), float(conf)))

        frame_number += 1

    cap.release()
    
    return pred_dict

In [8]:
model = YOLO('..\\yolov8s.pt')

100%|██████████| 21.5M/21.5M [00:00<00:00, 29.3MB/s]


## Evaluate Object Detection Performance on Camera 1

In [9]:
video_path_cam1 = "..\\dataset\\CarLA\\Camera_1\\video\\camera_1.mp4"


yolov8_pred_dict = run_yolov8(model, video_path_cam1)


0: 384x640 8 cars, 1 bus, 3 traffic lights, 44.4ms
Speed: 8.2ms preprocess, 44.4ms inference, 190.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 8 cars, 3 traffic lights, 7.7ms
Speed: 2.1ms preprocess, 7.7ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 8 cars, 3 traffic lights, 7.4ms
Speed: 1.6ms preprocess, 7.4ms inference, 1.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 8 cars, 3 traffic lights, 7.6ms
Speed: 1.7ms preprocess, 7.6ms inference, 2.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 7 cars, 3 traffic lights, 8.1ms
Speed: 1.6ms preprocess, 8.1ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 8 cars, 3 traffic lights, 7.9ms
Speed: 1.8ms preprocess, 7.9ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 8 cars, 3 traffic lights, 11.3ms
Speed: 2.0ms preprocess, 11.3ms inference, 1.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384

In [ ]:
gt_csv = "..\\dataset\\CarLA\\Camera_1\\bboxes.csv"
yolov7_pred_dir = "..\\..\\yolov7-segmentation\\runs\\predict-seg\\exp\\labels\\"

gt_dict = load_ground_truth(gt_csv)

print("Ground truth loaded successfully.")
print("-" * 40)
print("Evaluating YOLOv7-segmentation Results...")

yolov7_pred_dict = load_predictions(yolov7_pred_dir)

p_v7, r_v7, f1_v7 = evaluate_at_iou(gt_dict, yolov7_pred_dict, 0.5)
print(f"AP50  | Precision: {p_v7:.3f}, Recall: {r_v7:.3f}, F1-Score: {f1_v7:.3f}")

mAP_v7, _ = evaluate_map(gt_dict, yolov7_pred_dict)
print(f"mAP@[0.5:0.95]: {mAP_v7:.3f}")
print("-" * 40)


print("Evaluating YOLOv8 Results...")
if yolov8_pred_dict is not None:
    p_v8, r_v8, f1_v8 = evaluate_at_iou(gt_dict, yolov8_pred_dict, 0.5)
    print(f"AP50  | Precision: {p_v8:.3f}, Recall: {r_v8:.3f}, F1-Score: {f1_v8:.3f}")

    mAP_v8, _ = evaluate_map(gt_dict, yolov8_pred_dict)
    print(f"mAP@[0.5:0.95]: {mAP_v8:.3f}")
else:
    print("Skipping YOLOv8 evaluation because video processing failed. Check the video path.")
print("-" * 40)

## Evaluate Object Detection Performance on Camera 2

In [15]:
video_path_cam2 = "..\\dataset\\CarLA\\Camera_2\\video\\Camera_2.mp4"


yolov8_pred_dict = run_yolov8(model, video_path_cam2)


0: 384x640 16 cars, 1 truck, 1 traffic light, 108.1ms
Speed: 3.2ms preprocess, 108.1ms inference, 7.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 19 cars, 2 traffic lights, 87.6ms
Speed: 2.9ms preprocess, 87.6ms inference, 4.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 18 cars, 2 traffic lights, 41.8ms
Speed: 2.9ms preprocess, 41.8ms inference, 3.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 19 cars, 2 traffic lights, 10.6ms
Speed: 1.9ms preprocess, 10.6ms inference, 2.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 19 cars, 2 traffic lights, 12.4ms
Speed: 1.6ms preprocess, 12.4ms inference, 2.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 19 cars, 1 bus, 1 traffic light, 10.8ms
Speed: 2.0ms preprocess, 10.8ms inference, 2.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 19 cars, 1 bus, 1 traffic light, 10.1ms
Speed: 2.0ms preprocess, 10.1ms inference, 2.4ms postprocess per image at 

In [17]:
gt_csv = "..\\dataset\\CarLA\\Camera_2\\bboxes.csv"
yolov7_pred_dir = "..\\..\\yolov7-segmentation\\runs\\predict-seg\\exp2\\labels\\"

gt_dict = load_ground_truth(gt_csv)

print("Ground truth loaded successfully.")
print("-" * 40)
print("Evaluating YOLOv7-segmentation Results...")

yolov7_pred_dict = load_predictions(yolov7_pred_dir)

p_v7, r_v7, f1_v7 = evaluate_at_iou(gt_dict, yolov7_pred_dict, 0.5)
print(f"AP50  | Precision: {p_v7:.3f}, Recall: {r_v7:.3f}, F1-Score: {f1_v7:.3f}")

mAP_v7, _ = evaluate_map(gt_dict, yolov7_pred_dict)
print(f"mAP@[0.5:0.95]: {mAP_v7:.3f}")
print("-" * 40)


print("Evaluating YOLOv8 Results...")
if yolov8_pred_dict is not None:
    p_v8, r_v8, f1_v8 = evaluate_at_iou(gt_dict, yolov8_pred_dict, 0.5)
    print(f"AP50  | Precision: {p_v8:.3f}, Recall: {r_v8:.3f}, F1-Score: {f1_v8:.3f}")

    mAP_v8, _ = evaluate_map(gt_dict, yolov8_pred_dict)
    print(f"mAP@[0.5:0.95]: {mAP_v8:.3f}")
else:
    print("Skipping YOLOv8 evaluation because video processing failed. Check the video path.")
print("-" * 40)

Ground truth loaded successfully.
----------------------------------------
Evaluating YOLOv7-segmentation Results...
AP50  | Precision: 0.428, Recall: 0.512, F1-Score: 0.466
mAP@[0.5:0.95]: 0.194
----------------------------------------
Evaluating YOLOv8 Results...
AP50  | Precision: 0.606, Recall: 0.497, F1-Score: 0.546
mAP@[0.5:0.95]: 0.290
----------------------------------------


## Evaluate Object Detection Performance on Camera 3

In [18]:
video_path_cam3 = "..\\dataset\\CarLA\\Camera_3\\video\\Camera_3.mp4"


yolov8_pred_dict = run_yolov8(model, video_path_cam3)


0: 384x640 1 person, 6 cars, 1 truck, 1 traffic light, 70.3ms
Speed: 2.9ms preprocess, 70.3ms inference, 2.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 8 cars, 1 traffic light, 90.2ms
Speed: 3.8ms preprocess, 90.2ms inference, 2.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 8 cars, 1 traffic light, 81.8ms
Speed: 7.4ms preprocess, 81.8ms inference, 2.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 8 cars, 8.9ms
Speed: 1.8ms preprocess, 8.9ms inference, 1.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 8 cars, 8.5ms
Speed: 1.7ms preprocess, 8.5ms inference, 1.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 8 cars, 8.6ms
Speed: 1.7ms preprocess, 8.6ms inference, 2.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 8 cars, 7.7ms
Speed: 1.8ms preprocess, 7.7ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 8 cars, 7.6ms
Speed: 2.0ms preprocess,

In [19]:
gt_csv = "..\\dataset\\CarLA\\Camera_3\\bboxes.csv"
yolov7_pred_dir = "..\\..\\yolov7-segmentation\\runs\\predict-seg\\exp3\\labels\\"

gt_dict = load_ground_truth(gt_csv)

print("Ground truth loaded successfully.")
print("-" * 40)
print("Evaluating YOLOv7-segmentation Results...")

yolov7_pred_dict = load_predictions(yolov7_pred_dir)

p_v7, r_v7, f1_v7 = evaluate_at_iou(gt_dict, yolov7_pred_dict, 0.5)
print(f"AP50  | Precision: {p_v7:.3f}, Recall: {r_v7:.3f}, F1-Score: {f1_v7:.3f}")

mAP_v7, _ = evaluate_map(gt_dict, yolov7_pred_dict)
print(f"mAP@[0.5:0.95]: {mAP_v7:.3f}")
print("-" * 40)


print("Evaluating YOLOv8 Results...")
if yolov8_pred_dict is not None:
    p_v8, r_v8, f1_v8 = evaluate_at_iou(gt_dict, yolov8_pred_dict, 0.5)
    print(f"AP50  | Precision: {p_v8:.3f}, Recall: {r_v8:.3f}, F1-Score: {f1_v8:.3f}")

    mAP_v8, _ = evaluate_map(gt_dict, yolov8_pred_dict)
    print(f"mAP@[0.5:0.95]: {mAP_v8:.3f}")
else:
    print("Skipping YOLOv8 evaluation because video processing failed. Check the video path.")
print("-" * 40)

Ground truth loaded successfully.
----------------------------------------
Evaluating YOLOv7-segmentation Results...
AP50  | Precision: 0.950, Recall: 0.592, F1-Score: 0.729
mAP@[0.5:0.95]: 0.418
----------------------------------------
Evaluating YOLOv8 Results...
AP50  | Precision: 0.952, Recall: 0.560, F1-Score: 0.705
mAP@[0.5:0.95]: 0.429
----------------------------------------
